# WEEKLY TASK 3:

Build a complete text preprocessing and analysis pipeline using NLTK and spaCy. Given a dataset of 100+ news headlines, apply tokenization, POS tagging, NER, TF-IDF vectorization, and find the top similar headline pairs using cosine similarity. Write a 1-page summary of what each step

#  CNBC News Headlines — NLP Pipeline
### Text Preprocessing & Analysis using NLTK and spaCy

**Dataset:** Real CNBC news headlines (553 articles)

**Pipeline Steps:**
1. Load & explore the dataset
2. Tokenization
3. POS Tagging
4. Named Entity Recognition (NER)
5. TF-IDF Vectorization
6. Cosine Similarity — find most similar headlines

In [1]:
# installing required libraries

!pip install nltk spacy scikit-learn pandas
!python -m spacy download en_core_web_sm

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached pandas-3.0.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.5 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.5 kB ? eta -:--:--
     ------------------ ------------------- 20.5/41.5 kB 108.9 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 108.9 kB/s eta 0:00:01
     ---------------------------- --------- 30.7/41.5 kB 119.1 kB/s eta 0:00:01
     ---------------------------- --------- 30.7/41.5 kB 119.1 kB/s eta 0:00:01
     -------------------------------------  41.0/41.5 kB 131.3 kB/s eta 0:00:01
     -------------------------------------- 41.5/41.5 kB 111.2 kB/s eta 0:00:00
  Using cached scipy-1.17.1-cp311-c


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\lenevo\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.8 MB 4.7 MB/s eta 0:00:03
     --------- ------------------------------ 2.9/12.8 MB 6.5 MB/s eta 0:00:02
     ------------- -------------------------- 4.5/12.8 MB 6.2 MB/s eta 0:00:02
     ----------------- ---------------------- 5.5/12.8 MB 6.0 MB/s eta 0:00:02
     ------------------- -------------------- 6.3/12.8 MB 5.6 MB/s eta 0:00:02
     ----------------------- ---------------- 7.6/12.8 MB 5.6 MB/s eta 0:00:01
     -------------------------- ------------- 8.4/12.8 MB 5.4 MB/s eta 0:00:01
     ---------------------------- ----------- 9.2/12.8 MB 5.2 MB/s eta 0:00:01
     ------------------------------ --------- 9.7/12.8 MB 5.0 MB/s eta 0:00:01
     -------------------------------- ------- 10.5/12.8 MB 4.8 MB/s eta 0


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import nltk
import spacy
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# download required nltk data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')


#load spacy english model
nlp = spacy.load("en_core_web_sm")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lenevo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lenevo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\lenevo\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\lenevo\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


##  Load the CNBC Dataset

We load our real CNBC CSV file.
The important column is **`title`** — that's our headline text.
We also keep `category` and `author` for extra context.

In [3]:
# Load the CSV file 
df = pd.read_csv("cnbc_news_data.csv")

# Keep only the useful columns
df = df[['title', 'category', 'author', 'published_at']].copy()

# Drop rows where title is missing
df = df.dropna(subset=['title']).reset_index(drop=True)

print(f"  Dataset loaded!")
print(f"   Total headlines : {len(df)}")
print(f"   Columns         : {df.columns.tolist()}")
print()
df.head(5)

  Dataset loaded!
   Total headlines : 553
   Columns         : ['title', 'category', 'author', 'published_at']



,title,category,author,published_at
0,iPhone maker Foxconn to invest $600 million in...,Technology,Arjun Kharpal,2023-08-02T13:31:01+0000
1,Education Department investigating Harvard aft...,Politics,Mirna Alsharif and Doha Madani,2024-02-07T03:40:25+0000
2,Analysts Speak on Today’s Market,Business News,By CNBC.com,2007-03-26T19:19:24+0000
3,New Jersey Borrowing a Sign of Coming Muni Cri...,Business News,Jeff Cox,2011-06-28T13:51:16+0000
4,Fear Miners Set to Fall Off 'Supercycle',Business News,Jack Farchy and Helen Thomas,2012-05-10T10:44:43+0000


###  Explore the Dataset

In [5]:
print(" Headline count by Category:\n")
print(df['category'].value_counts().to_string())
print()
print(" Date range of articles:")
print("  Earliest:", df['published_at'].min())
print("  Latest  :", df['published_at'].max())

 Headline count by Category:

category
Business News                    148
Markets                           81
CNBC TV                           64
Technology                        48
Politics                          41
Investing                         35
Wires                             32
CNBC Information and Policies     22
PRO Home                          22
CNBC Investing Club                7
CNBC EVENTS                        7
Opinion                            5
Special Reports                    5
Asia Economy                       4
Companies                          4
Sports Biz with Darren Rovell      3
U.S. Markets Top News              3
Money in Motion                    2
Tech Check                         2
do not use_28258269                2
Slide shows                        2
Business Sectors                   1
Mike on America                    1
China: Time of Transition          1
What Investors Should Know         1
Investor Agenda                    1

In [6]:
print(" Sample Headlines from each Category:\n")

for category in df['category'].dropna().unique()[:5]:  # show first 5 categories
    sample = df[df['category'] == category]['title'].iloc[0]
    print(f"[{category}]")
    print(f"  → {sample}")
    print()

 Sample Headlines from each Category:

[Technology]
  → iPhone maker Foxconn to invest $600 million into phone and chip project in India

[Politics]
  → Education Department investigating Harvard after complaint from Palestinian students and allies

[Business News]
  → Analysts Speak on Today’s Market 

[CNBC Investing Club]
  → Here's what investors need to know about the strong dollar's impact on our stocks

[Markets]
  → Oil markets' decades-long dependence on China could be ending



##  Tokenization

**What is it?**
Tokenization splits each headline into individual words, called "tokens".

**Example:**
- Input:  `"Foxconn to invest $600 million in India"`
- Output: `['Foxconn', 'to', 'invest', '$', '600', 'million', 'in', 'India']`

**Why?**

Computers need words as separate units, not full sentences: to do any NLP analysis.
We use NLTK's `word_tokenize()` function.

In [7]:
from nltk.tokenize import word_tokenize

# Apply tokenization to every headline
df['tokens'] = df['title'].apply(word_tokenize)

# Show results for first 5 headlines
print(" Tokenization Results:\n")
for i in range(5):
    print(f"Original : {df['title'][i]}")
    print(f"Tokens   : {df['tokens'][i]}")
    print("-" * 65)

 Tokenization Results:

Original : iPhone maker Foxconn to invest $600 million into phone and chip project in India
Tokens   : ['iPhone', 'maker', 'Foxconn', 'to', 'invest', '$', '600', 'million', 'into', 'phone', 'and', 'chip', 'project', 'in', 'India']
-----------------------------------------------------------------
Original : Education Department investigating Harvard after complaint from Palestinian students and allies
Tokens   : ['Education', 'Department', 'investigating', 'Harvard', 'after', 'complaint', 'from', 'Palestinian', 'students', 'and', 'allies']
-----------------------------------------------------------------
Original : Analysts Speak on Today’s Market 
Tokens   : ['Analysts', 'Speak', 'on', 'Today', '’', 's', 'Market']
-----------------------------------------------------------------
Original : New Jersey Borrowing a Sign of Coming Muni Crisis: Whitney
Tokens   : ['New', 'Jersey', 'Borrowing', 'a', 'Sign', 'of', 'Coming', 'Muni', 'Crisis', ':', 'Whitney']
-----------

In [8]:
# Count tokens per headline
df['token_count'] = df['tokens'].apply(len)

print(" Token Statistics:")
print(f"  Average tokens per headline : {df['token_count'].mean():.1f}")
print(f"  Max tokens in a headline    : {df['token_count'].max()}")
print(f"  Min tokens in a headline    : {df['token_count'].min()}")
print()
print(" Longest headline:")
print(" ", df.loc[df['token_count'].idxmax(), 'title'])

 Token Statistics:
  Average tokens per headline : 11.3
  Max tokens in a headline    : 32
  Min tokens in a headline    : 1

 Longest headline:
  CNBC launches new segment series on Clean energy. ‘People, Planet, Profit’ to air on CNBC around the world, in association with The Zayed Future Energy Prize


##  POS Tagging (Part-of-Speech Tagging)

**What is it?**
POS tagging labels each word with its grammatical role.

| Tag | Meaning | Example |
|-----|---------|---------|
| NNP | Proper Noun | Foxconn, India, Harvard |
| VBZ | Verb (present) | invests, rises, falls |
| JJ  | Adjective | strong, new, major |
| NN  | Common Noun | market, deal, price |
| IN  | Preposition | in, of, for |

**Why?**
It helps us understand *what* words do in a sentence.
Useful for extracting key nouns, finding verbs (actions), and understanding grammar.

In [9]:
from nltk import pos_tag

# Apply POS tagging to tokens
df['pos_tags'] = df['tokens'].apply(pos_tag)

# Show results for first 5 headlines
print(" POS Tagging Results:\n")
for i in range(5):
    print(f"Headline : {df['title'][i]}")
    print(f"POS Tags : {df['pos_tags'][i]}")
    print("-" * 65)

 POS Tagging Results:

Headline : iPhone maker Foxconn to invest $600 million into phone and chip project in India
POS Tags : [('iPhone', 'NN'), ('maker', 'NN'), ('Foxconn', 'NNP'), ('to', 'TO'), ('invest', 'VB'), ('$', '$'), ('600', 'CD'), ('million', 'CD'), ('into', 'IN'), ('phone', 'NN'), ('and', 'CC'), ('chip', 'NN'), ('project', 'NN'), ('in', 'IN'), ('India', 'NNP')]
-----------------------------------------------------------------
Headline : Education Department investigating Harvard after complaint from Palestinian students and allies
POS Tags : [('Education', 'NNP'), ('Department', 'NNP'), ('investigating', 'VBG'), ('Harvard', 'NNP'), ('after', 'IN'), ('complaint', 'NN'), ('from', 'IN'), ('Palestinian', 'JJ'), ('students', 'NNS'), ('and', 'CC'), ('allies', 'NNS')]
-----------------------------------------------------------------
Headline : Analysts Speak on Today’s Market 
POS Tags : [('Analysts', 'NNS'), ('Speak', 'VBP'), ('on', 'IN'), ('Today', 'NNP'), ('’', 'NNP'), ('s', 'NN

In [10]:
from collections import Counter

# Collect all POS tags from all headlines
all_tags = []
for tag_list in df['pos_tags']:
    for word, tag in tag_list:
        all_tags.append(tag)

tag_counts = Counter(all_tags)

print(" Top 10 Most Common POS Tags in CNBC Headlines:\n")
for tag, count in tag_counts.most_common(10):
    print(f"  {tag:6} → {count} times")

 Top 10 Most Common POS Tags in CNBC Headlines:

  NNP    → 1595 times
  NN     → 874 times
  IN     → 559 times
  NNS    → 509 times
  JJ     → 378 times
  DT     → 238 times
  VBZ    → 214 times
  VB     → 191 times
  ,      → 187 times
  CD     → 173 times


## Named Entity Recognition (NER)

**What is it?**
NER finds and labels real-world "named entities" in text.

| Label | Meaning | Example from CNBC |
|-------|---------|-------------------|
| ORG | Organization | Foxconn, Harvard, CNBC |
| GPE | Country/City | India, New Jersey, China |
| PERSON | Person name | Meredith Whitney, Jeff Cox |
| MONEY | Money amount | $600 million, $2.25 billion |
| DATE | Date/time | Tuesday, last year, 2023 |

**Why?**
NER extracts structured facts from raw headlines.
Journalists, analysts, and AI search engines all use NER to understand *who*, *where*, and *what* is in a story.

We use **spaCy** : it's faster and more accurate than NLTK for NER.

In [12]:
# Function to extract entities from a headline
def get_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

# Apply NER to all headlines
df['entities'] = df['title'].apply(get_entities)

# Show results for first 5 headlines
print(" NER Results:\n")
for i in range(10):
    print(f"Headline : {df['title'][i]}")
    print(f"Entities : {df['entities'][i]}")
    print("-" * 65)

 NER Results:

Headline : iPhone maker Foxconn to invest $600 million into phone and chip project in India
Entities : [('Foxconn', 'GPE'), ('$600 million', 'MONEY'), ('India', 'GPE')]
-----------------------------------------------------------------
Headline : Education Department investigating Harvard after complaint from Palestinian students and allies
Entities : [('Education Department', 'ORG'), ('Harvard', 'ORG'), ('Palestinian', 'NORP')]
-----------------------------------------------------------------
Headline : Analysts Speak on Today’s Market 
Entities : []
-----------------------------------------------------------------
Headline : New Jersey Borrowing a Sign of Coming Muni Crisis: Whitney
Entities : [('New Jersey Borrowing', 'GPE')]
-----------------------------------------------------------------
Headline : Fear Miners Set to Fall Off 'Supercycle'
Entities : []
-----------------------------------------------------------------
Headline : Here's what investors need to know abo

In [13]:
from spacy import displacy

# Visually highlight entities in first 4 headlines
print(" Visual NER — entities highlighted by type:\n")
for i in range(4):
    doc = nlp(df['title'][i])
    print(f"Headline {i+1}: {df['title'][i]}")
    displacy.render(doc, style="ent", jupyter=True)
    print()

 Visual NER — entities highlighted by type:

Headline 1: iPhone maker Foxconn to invest $600 million into phone and chip project in India



Headline 2: Education Department investigating Harvard after complaint from Palestinian students and allies



Headline 3: Analysts Speak on Today’s Market 


c:\Users\lenevo\AppData\Local\Programs\Python\Python313\Lib\site-packages\spacy\displacy\__init__.py:214: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)



Headline 4: New Jersey Borrowing a Sign of Coming Muni Crisis: Whitney


In [14]:
from collections import Counter

# Flatten all entities into one list
all_entities = []
for ent_list in df['entities']:
    for text, label in ent_list:
        all_entities.append((text, label))

entity_counts = Counter(all_entities)

print(" Top 15 Most Mentioned Named Entities in CNBC Headlines:\n")
for (entity, label), count in entity_counts.most_common(15):
    print(f"  {entity:30} [{label}]  → {count}x")

 Top 15 Most Mentioned Named Entities in CNBC Headlines:

  US                             [GPE]  → 19x
  China                          [GPE]  → 16x
  Trump                          [ORG]  → 12x
  Apple                          [ORG]  → 11x
  Fed                            [ORG]  → 10x
  CNBC                           [ORG]  → 9x
  3                              [CARDINAL]  → 8x
  Treasury                       [ORG]  → 8x
  Europe                         [LOC]  → 7x
  U.S.                           [GPE]  → 6x
  2                              [CARDINAL]  → 6x
  One                            [CARDINAL]  → 5x
  Trump                          [PERSON]  → 5x
  5                              [CARDINAL]  → 5x
  GOP                            [ORG]  → 5x


##  TF-IDF Vectorization

**What is TF-IDF?**
TF-IDF = **Term Frequency × Inverse Document Frequency**

It gives each word a score that reflects how *important* it is:

- **High score** → word appears in a few headlines but is meaningful (e.g. "Foxconn", "inflation")
- **Low score** → word appears everywhere, not meaningful (e.g. "the", "a", "is")

**Why?**
TF-IDF converts text into numbers (vectors).
Without numbers, we can't do math — and we need math to compare headlines.

`stop_words='english'` automatically removes common words like "the", "a", "in".

In [15]:
# Build TF-IDF matrix from all headlines
vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
tfidf_matrix = vectorizer.fit_transform(df['title'])

feature_names = vectorizer.get_feature_names_out()

print(f" TF-IDF Matrix created!")
print(f"   Shape: {tfidf_matrix.shape}")
print(f"   → {tfidf_matrix.shape[0]} headlines × {tfidf_matrix.shape[1]} unique words")
print(f"\n Sample vocabulary: {list(feature_names[:20])}")

 TF-IDF Matrix created!
   Shape: (553, 500)
   → 553 headlines × 500 unique words

 Sample vocabulary: ['000', '10', '15', '17', '20', '2008', '2018', '2020', '2021', '2022', '2023', '2024', '50', 'ahead', 'ai', 'air', 'airlines', 'alert', 'amazon', 'america']


In [16]:
import numpy as np

print(" Top 5 Important Words per Headline (by TF-IDF score):\n")

for i in range(5):
    # Get TF-IDF scores for this headline
    row = tfidf_matrix[i].toarray()[0]
    
    # Get indices of top 5 scores
    top_indices = row.argsort()[::-1][:5]
    top_words = [(feature_names[j], round(row[j], 3)) for j in top_indices if row[j] > 0]
    
    print(f"Headline : {df['title'][i]}")
    print(f"Top Words: {top_words}")
    print("-" * 65)

 Top 5 Important Words per Headline (by TF-IDF score):

Headline : iPhone maker Foxconn to invest $600 million into phone and chip project in India
Top Words: [('maker', np.float64(0.426)), ('project', np.float64(0.426)), ('invest', np.float64(0.426)), ('india', np.float64(0.426)), ('iphone', np.float64(0.386))]
-----------------------------------------------------------------
Headline : Education Department investigating Harvard after complaint from Palestinian students and allies
Top Words: [('education', np.float64(1.0))]
-----------------------------------------------------------------
Headline : Analysts Speak on Today’s Market 
Top Words: [('speak', np.float64(0.591)), ('today', np.float64(0.511)), ('analysts', np.float64(0.487)), ('market', np.float64(0.39))]
-----------------------------------------------------------------
Headline : New Jersey Borrowing a Sign of Coming Muni Crisis: Whitney
Top Words: [('sign', np.float64(0.572)), ('coming', np.float64(0.525)), ('crisis', np.f

##  Cosine Similarity

**What is it?**
Cosine similarity measures how similar two headlines are, based on their TF-IDF word patterns.

- **Score = 1.0** → headlines are nearly identical in topic
- **Score = 0.5** → some shared topic/words
- **Score = 0.0** → completely different topics

**Formula (simple version):**
> *How much do the word-vectors of two headlines point in the same direction?*

**Why?**
News agencies use this to:
- Detect duplicate or near-duplicate articles
- Cluster related stories together
- Build "Related Articles" recommendations

We compare **every pair** of headlines and find the top 10 most similar pairs.

# computing similarity

In [19]:
# Compute cosine similarity between all headline pairs
similarity_matrix = cosine_similarity(tfidf_matrix)

print(f" Similarity Matrix Shape: {similarity_matrix.shape}")
print(f"   ({similarity_matrix.shape[0]} × {similarity_matrix.shape[1]} pairs computed)")
print(f"\n   Example: Similarity between headline 0 and 1 = {similarity_matrix[0][1]:.4f}")

print(f"\n   Example: Similarity between headline 2 and 7 = {similarity_matrix[2][7]:.4f}")

 Similarity Matrix Shape: (553, 553)
   (553 × 553 pairs computed)

   Example: Similarity between headline 0 and 1 = 0.0000

   Example: Similarity between headline 2 and 7 = 0.0000


# finding out top similar pairs

In [25]:
# Collect all unique pairs with similarity score > 0
pairs = []

for i in range(len(df)):
    for j in range(i + 1, len(df)):
        score = similarity_matrix[i][j]
        if score > 0:
            pairs.append({
                'Headline 1': df['title'][i],
                'Category 1': df['category'][i],
                'Headline 2': df['title'][j],
                'Category 2': df['category'][j],
                'Similarity': round(score, 4)
            })

# Sort by similarity score
pairs_df = pd.DataFrame(pairs)
pairs_df = pairs_df.sort_values('Similarity', ascending=False).reset_index(drop=True)

print(f" Total similar pairs found: {len(pairs_df)}")
print(f"\n Preview of top 5 pairs:")
pairs_df[['Headline 1', 'Headline 2', 'Similarity']].head(20)

 Total similar pairs found: 6476

 Preview of top 5 pairs:


,Headline 1,Headline 2,Similarity
0,No. 2 - Search For A Bottom In Yahoo?,A Creative Approach to Your Job Search,1.0000
1,"Lightning Round OT: Sprint Nextel, Lionsgate a...","Lightning Round OT: Immunogen, Paychex and More",1.0000
2,Tax Cheats Get Extended Reprieve from IRS,Your tax return could be flagged by the IRS. H...,1.0000
3,Will China’s slowdown derail reforms?,China's Xi to hold talks with Indonesia's Joko...,1.0000
4,Lightning Round: I missed it on this one,Lightning Round: It's ridiculously overvalued,1.0000
5,Clashes break out as Hong Kong protesters esca...,Hong Kong stocks: It's now or never,0.9113
6,Stocks making the biggest moves after hours: T...,Stocks making the biggest moves after hours: H...,0.8491
7,Stocks making the biggest moves midday: Novava...,Stocks making the biggest moves after hours: T...,0.8491
8,CCTV Script 29/06/20,CCTV Script 13/08/18,0.8421
9,RESEARCH ALERT-BMO cuts Aureus Mining price ta...,RESEARCH ALERT-Wedbush cuts Bebe Stores price ...,0.8278


Print Top 10 Pairs 

In [26]:
print(" TOP 10 MOST SIMILAR CNBC HEADLINE PAIRS")


for idx, row in pairs_df.head(10).iterrows():
    print(f"\n#{idx+1}  Similarity Score: {row['Similarity']}")
    print(f"   [{row['Category 1']}] {row['Headline 1']}")
    print(f"   [{row['Category 2']}] {row['Headline 2']}")
    

 TOP 10 MOST SIMILAR CNBC HEADLINE PAIRS

#1  Similarity Score: 1.0
   [CNBC TV] No. 2 - Search For A Bottom In Yahoo?
   [Executive Careers] A Creative Approach to Your Job Search

#2  Similarity Score: 1.0
   [CNBC TV] Lightning Round OT: Sprint Nextel, Lionsgate and More
   [CNBC TV] Lightning Round OT: Immunogen, Paychex and More

#3  Similarity Score: 1.0
   [Business News] Tax Cheats Get Extended Reprieve from IRS 
   [Investing] Your tax return could be flagged by the IRS. Here’s when it may happen

#4  Similarity Score: 1.0
   [Business News] Will China’s slowdown derail reforms?
   [Politics] China's Xi to hold talks with Indonesia's Jokowi in rare visit

#5  Similarity Score: 1.0
   [CNBC TV] Lightning Round: I missed it on this one
   [CNBC TV] Lightning Round: It's ridiculously overvalued

#6  Similarity Score: 0.9113
   [Politics] Clashes break out as Hong Kong protesters escalate fight
   [Markets] Hong Kong stocks: It's now or never

#7  Similarity Score: 0.8491
   [Mark

# Category-wise Similarity

In [28]:
print(" Do similar headlines tend to share the same category?\n")

same_cat = pairs_df[pairs_df['Category 1'] == pairs_df['Category 2']]
diff_cat = pairs_df[pairs_df['Category 1'] != pairs_df['Category 2']]

print(f"  Pairs from SAME category     : {len(same_cat)}")
print(f"  Pairs from DIFFERENT category: {len(diff_cat)}")
print()
print(f"  Avg similarity (same category)    : {same_cat['Similarity'].mean():.4f}")
print(f"  Avg similarity (diff category)    : {diff_cat['Similarity'].mean():.4f}")

 Do similar headlines tend to share the same category?

  Pairs from SAME category     : 1311
  Pairs from DIFFERENT category: 5165

  Avg similarity (same category)    : 0.2312
  Avg similarity (diff category)    : 0.1980


# Final Summary Statistics

In [30]:
df['entity_count'] = df['entities'].apply(len)

print("=" * 55)
print(" FULL PIPELINE SUMMARY — CNBC HEADLINES DATASET")
print("=" * 55)
print(f"  Total headlines analyzed     : {len(df)}")
print(f"  News categories              : {df['category'].nunique()}")
print(f"  Avg tokens per headline      : {df['token_count'].mean():.1f}")
print(f"  Total unique vocab (TF-IDF)  : {len(feature_names)}")
print(f"  Avg named entities/headline  : {df['entity_count'].mean():.1f}")
print(f"  Total similar pairs found    : {len(pairs_df)}")
print(f"  Top similarity score         : {pairs_df['Similarity'].max()}")
print()

print(" Top 5 headlines with most named entities:")
top_ent = df[['title', 'entity_count']].sort_values('entity_count', ascending=False).head(5)
for _, row in top_ent.iterrows():
    print(f"  ({row['entity_count']} entities) {row['title']}")

 FULL PIPELINE SUMMARY — CNBC HEADLINES DATASET
  Total headlines analyzed     : 553
  News categories              : 32
  Avg tokens per headline      : 11.3
  Total unique vocab (TF-IDF)  : 500
  Avg named entities/headline  : 1.7
  Total similar pairs found    : 6476
  Top similarity score         : 1.0

 Top 5 headlines with most named entities:
  (9 entities) First on CNBC: CNBC Transcript: Square CFO Amrita Ahuja and Afterpay Co-Founder & Co-CEO Nick Molnar Speak with CNBC’s “Squawk on the Street” Today
  (7 entities) CNBC Excerpts: CNBC’s Sara Eisen Speaks with Former Fed Chairman Alan Greenspan & IMF Managing Director Christine Lagarde Today
  (6 entities) Here are Tuesday's biggest analyst calls: Apple, Tesla, Dick's, Costco, Snowflake, Meta, Dish & more
  (5 entities) Early movers: PBY, NKE, BBY, JCP, H, NFLX, GMCR & more
  (5 entities) Sheldon and Miriam Adelson give $25 million to help Republicans keep their majority in the Senate


In [31]:
print(" Final DataFrame (first 5 rows):\n")
df[['title', 'category', 'token_count', 'entity_count', 'entities']].head(5)

 Final DataFrame (first 5 rows):



,title,category,token_count,entity_count,entities
0,iPhone maker Foxconn to invest $600 million in...,Technology,15,3,"[(Foxconn, GPE), ($600 million, MONEY), (India..."
1,Education Department investigating Harvard aft...,Politics,11,3,"[(Education Department, ORG), (Harvard, ORG), ..."
2,Analysts Speak on Today’s Market,Business News,7,0,[]
3,New Jersey Borrowing a Sign of Coming Muni Cri...,Business News,11,1,"[(New Jersey Borrowing, GPE)]"
4,Fear Miners Set to Fall Off 'Supercycle',Business News,8,0,[]


---
##  NLP Pipeline on CNBC Headlines

---

### 1. Tokenization
**What:** Split each headline into individual words (tokens) using NLTK's `word_tokenize()`.
**Why:** Text must be broken into units before any NLP can happen. Tokenization is Step 1 of every NLP pipeline.

### 2. POS Tagging
**What:** Each token is labeled with a grammatical role — noun, verb, adjective, etc. — using NLTK's `pos_tag()`.
**Why:** Understanding grammar structure helps identify the most meaningful words. In CNBC headlines, proper nouns (NNP) and nouns (NN) carry the most information.

### 3. Named Entity Recognition (NER)
**What:** spaCy identifies real-world entities: companies (ORG), people (PERSON), locations (GPE), money (MONEY), and dates (DATE).
**Why:** NER extracts structured facts from unstructured text. In finance news, spotting "Foxconn", "$600 million", and "India" is far more useful than reading the full sentence.

### 4. TF-IDF Vectorization
**What:** Each headline is converted into a numerical vector. Words that are unique to a small number of articles get higher scores; common words like "the" and "a" are automatically filtered out.
**Why:** Computers cannot compare raw text — they need numbers. TF-IDF is the standard method to represent documents mathematically while preserving meaning.

### 5. Cosine Similarity
**What:** The angle between two TF-IDF vectors is measured. A score close to 1.0 means the two headlines share similar vocabulary and topic; 0.0 means no overlap.
**Why:** This is how "Related Articles" features work on news websites. Instead of reading every article, the algorithm instantly finds pairs that cover similar topics.

---
### Key Finding
Most highly similar CNBC headline pairs came from the **same category** (e.g. two Business News headlines about markets), confirming that TF-IDF + cosine similarity successfully groups topic-related content — without any human labeling.